# ReWear Sustainability-First Fine-Tuning

This notebook fine-tunes a sustainability-first instruction model on rule-generated wardrobe prompts, then evaluates the fine-tuned and untuned base models on the same frozen held-out prompt set.

The evaluation saves full prompt-level outputs, automated metric flags, and aggregate results. The style-first JSONL file is retained for dataset documentation but is **not** evaluated as a third model condition in this notebook.

## 1. Setup

In [ ]:
# Choose the model identifier already approved for my account.
USE_LLAMA = True

MODEL_NAME = (
    "unsloth/llama-3-8b-Instruct-bnb-4bit" if USE_LLAMA
    else "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
)
print("Model:", MODEL_NAME)

Model: unsloth/llama-3-8b-Instruct-bnb-4bit


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Record the exact installed package versions for this run. The pip
# install command above uses loose constraints (e.g. trl<0.9.0)
!pip freeze | grep -Ei "^(unsloth|trl|transformers|torch|peft|accelerate|bitsandbytes|xformers)=="


accelerate==1.14.0
bitsandbytes==0.50.2
peft==0.20.0
torch==2.11.0+cu128
transformers==5.5.0
trl==0.24.0


In [ ]:
# Only needed when the selected model requires authenticated access.
if USE_LLAMA:
    from huggingface_hub import login
    login()
else:
    print("No login required for the selected model.")

## 2. Upload and prepare JSONL data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/prompt_response_pairs.jsonl" .
!cp "/content/drive/MyDrive/baseline_prompt_response_pairs.jsonl" .

In [ ]:
!ls -la prompt_response_pairs.jsonl baseline_prompt_response_pairs.jsonl

-rw------- 1 root root 1082100 Aug 30 15:57 baseline_prompt_response_pairs.jsonl
-rw------- 1 root root 1537609 Aug 30 15:57 prompt_response_pairs.jsonl


In [ ]:
import json

def load_pairs(path: str) -> list[dict]:
    """Load JSONL robustly, including objects written back-to-back."""
    text = open(path, encoding="utf-8").read().strip()
    pairs, decoder, idx, n = [], json.JSONDecoder(), 0, len(text)

    while idx < n:
        while idx < n and text[idx] in " \r\n\t":
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        pairs.append(obj)
        idx = end
    return pairs

def dedup(pairs: list[dict]) -> list[dict]:
    """Remove exact-duplicate (instruction, output) pairs, keeping first occurrence order."""
    seen, output = set(), []
    for pair in pairs:
        key = (pair.get("instruction", ""), pair.get("output", ""))
        if key not in seen:
            seen.add(key)
            output.append(pair)
    return output

sust = dedup(load_pairs("prompt_response_pairs.jsonl"))
style_first = dedup(load_pairs("baseline_prompt_response_pairs.jsonl"))

print(f"Sustainability-first pairs: {len(sust)}")
print(f"Style-first pairs retained for documentation: {len(style_first)}")


Sustainability-first pairs: 600
Style-first pairs retained for documentation: 600


## 3. Frozen train, validation, and held-out test split

In [ ]:
import random

SPLIT_SEED = 42
random.seed(SPLIT_SEED)
random.shuffle(sust)

n = len(sust)
n_test = min(50, max(20, n // 10))
n_val = min(50, max(20, n // 10))

test_set = sust[:n_test]
val_set = sust[n_test:n_test + n_val]
train_set = sust[n_test + n_val:]

print(f"Train: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

# Persist the exact held-out examples used for every evaluation rerun.
with open("heldout_test_set.jsonl", "w", encoding="utf-8") as f:
    for prompt_id, example in enumerate(test_set):
        record = {"prompt_id": prompt_id, **example}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

Train: 500 | Validation: 50 | Test: 50


In [ ]:
from google.colab import files
files.download("heldout_test_set.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

audit_rows = []

for split_name, split_data in [
    ("train", train_set),
    ("validation", val_set),
    ("heldout_test", test_set),
]:
    for row_number, pair in enumerate(split_data):
        audit_rows.append({
            "split": split_name,
            "row_in_split": row_number,
            "instruction": pair.get("instruction", ""),
            "reference_output": pair.get("output", ""),
        })

audit_df = pd.DataFrame(audit_rows)

print(audit_df["split"].value_counts())
assert len(audit_df) == 600, "Expected 600 total records."

audit_filename = "split_integrity_audit.csv"
audit_df.to_csv(audit_filename, index=False, encoding="utf-8")

files.download(audit_filename)
print(f"Downloaded: {audit_filename}")


split
train           500
validation       50
heldout_test     50
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: split_integrity_audit.csv


## 4. Load the model and attach parameter-efficient adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model and adapters ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.22 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model and adapters ready.


## 5. Format the sustainability-first training data

In [ ]:
SYSTEM_PROMPT = (
    "You are a sustainability-first fashion stylist. From the user's existing "
    "wardrobe only, recommend a complete outfit that maximises reuse of "
    "already-owned garments and favours lower-impact materials, and explain "
    "why it is the more sustainable choice. Never suggest buying anything new."
)

def to_chat(example: dict) -> dict:
    """Render one instruction/output pair into the model's chat template,
    ready for supervised fine-tuning."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

from datasets import Dataset
train_ds = Dataset.from_list([to_chat(example) for example in train_set])
val_ds = Dataset.from_list([to_chat(example) for example in val_set])
print(train_ds[0]["text"][:400])


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a sustainability-first fashion stylist. From the user's existing wardrobe only, recommend a complete outfit that maximises reuse of already-owned garments and favours lower-impact materials, and explain why it is the more sustainable choice. Never suggest buying anything new.<|eot_id|><|start_header_id|>user<|end_header_id|>

Use


## 6. Fine-tune

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_ft",
        report_to="none",
    ),
)

train_output = trainer.train()
print(train_output)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 2 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.612389
20,0.812763
30,0.674380
40,0.638921
50,0.621214
60,0.607118
70,0.578029
80,0.572693
90,0.564786
100,0.547668


Unsloth: Restored added_tokens_decoder metadata in outputs_ft/checkpoint-126/tokenizer_config.json.


TrainOutput(global_step=126, training_loss=0.6858295978061737, metrics={'train_runtime': 3221.8387, 'train_samples_per_second': 0.31, 'train_steps_per_second': 0.039, 'total_flos': 3.5839712514637824e+16, 'train_loss': 0.6858295978061737, 'epoch': 2.0})


In [ ]:
# Save adapters and tokenizer.
model.save_pretrained("rewear_sustainability_lora")
tokenizer.save_pretrained("rewear_sustainability_lora")
!zip -r rewear_sustainability_lora.zip rewear_sustainability_lora >/dev/null

from google.colab import files
files.download("rewear_sustainability_lora.zip")
print("Saved adapters.")


Unsloth: Restored added_tokens_decoder metadata in rewear_sustainability_lora/tokenizer_config.json.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved adapters.


## 7. Corrected held-out evaluation: fine-tuned model versus fresh base model

This section explicitly passes an attention mask, uses a single generation-length control, uses deterministic decoding, stores all prompt-level outputs, and writes aggregate comparison results. The automatic metrics are heuristics and require manual review of all new-purchase flags before dissertation reporting.

In [ ]:
# Load a fresh untuned base model for the comparison.
FastLanguageModel.for_inference(model)

base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print("Fine-tuned and fresh base models are ready for evaluation.")

==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Fine-tuned and fresh base models are ready for evaluation.


In [ ]:
import re
from typing import Any

import pandas as pd
from pathlib import Path

MAX_NEW_TOKENS = 200
OUTPUT_DIR = Path("evaluation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def build_generation_inputs(tok: Any, prompt: str) -> dict:
    """Create input IDs and an explicit attention mask."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    rendered = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # The chat template already inserts special tokens, so do not add them again.
    encoded = tok(
        rendered,
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length,
        add_special_tokens=False,
    )
    return {key: value.to("cuda") for key, value in encoded.items()}

@torch.inference_mode()
def generate_response(active_model: Any, active_tokenizer: Any, prompt: str) -> str:
    """Deterministic generation with an attention mask and one length limit."""
    encoded = build_generation_inputs(active_tokenizer, prompt)
    prompt_length = encoded["input_ids"].shape[1]

    output_ids = active_model.generate(
        **encoded,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=active_tokenizer.eos_token_id,
    )

    generated_ids = output_ids[0, prompt_length:]
    return active_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

SUSTAIN_WORDS = [
    "sustain", "reuse", "lower-impact", "already", "own",
    "new", "environment", "worn", "material",
]

POSITIVE_NEW_PURCHASE = re.compile(
    r"\b(?:buy|purchase)(?:\s+(?:a|any))?\s+new\b",
    flags=re.IGNORECASE,
)

NEGATED_NEW_PURCHASE = re.compile(
    r"\b(?:do not|don't|dont|not|never|avoid|rather than|instead of|"
    r"no need to)\s+(?:buy|purchase)(?:\s+(?:a|any))?\s+new\b",
    flags=re.IGNORECASE,
)

def recommends_buying_new(response: str) -> bool:
    """Flag explicit new-purchase recommendations, not negated warnings."""
    cleaned = NEGATED_NEW_PURCHASE.sub("", response)
    return bool(POSITIVE_NEW_PURCHASE.search(cleaned))

def score_response(prompt: str, response: str) -> dict:
    """Compute automated screening metrics for one generated response:
    whether it references an existing wardrobe item, whether it appears to
    recommend a new purchase, and how many distinct sustainability-related
    keywords it uses. These are coarse heuristics, not a quality judgement"""
    response_lower = response.lower()

    wardrobe_items = [
        line.split("[")[0].strip("- ").strip().lower()
        for line in prompt.splitlines()
        if line.strip().startswith("-")
    ]

    reuse = any(
        item[:15] in response_lower
        for item in wardrobe_items
        if len(item) > 4
    )

    buys_new = recommends_buying_new(response)

    # Counts distinct vocabulary strings that occur at least once.
    sustain_matches = sum(word in response_lower for word in SUSTAIN_WORDS)

    return {
        "reuse": bool(reuse),
        "buys_new": bool(buys_new),
        "sustainability_keyword_matches": int(sustain_matches),
    }

records = []

for prompt_id, example in enumerate(test_set):
    prompt = example["instruction"]

    fine_tuned_response = generate_response(model, tokenizer, prompt)
    base_response = generate_response(base_model, base_tok, prompt)

    fine_tuned_metrics = score_response(prompt, fine_tuned_response)
    base_metrics = score_response(prompt, base_response)

    records.append({
        "prompt_id": prompt_id,
        "prompt": prompt,
        "reference_target": example.get("output", ""),
        "fine_tuned_response": fine_tuned_response,
        "base_response": base_response,
        "fine_tuned_reuse": fine_tuned_metrics["reuse"],
        "fine_tuned_automated_buys_new": fine_tuned_metrics["buys_new"],
        "fine_tuned_manual_buys_new": "",
        "fine_tuned_sustainability_keyword_matches":
            fine_tuned_metrics["sustainability_keyword_matches"],
        "base_reuse": base_metrics["reuse"],
        "base_automated_buys_new": base_metrics["buys_new"],
        "base_manual_buys_new": "",
        "base_sustainability_keyword_matches":
            base_metrics["sustainability_keyword_matches"],
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "attention_mask_passed": True,
    })

per_prompt_df = pd.DataFrame(records)

per_prompt_csv = OUTPUT_DIR / "heldout_per_prompt_evaluation.csv"
per_prompt_jsonl = OUTPUT_DIR / "heldout_per_prompt_evaluation.jsonl"
per_prompt_df.to_csv(per_prompt_csv, index=False)

with open(per_prompt_jsonl, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

comparison = pd.DataFrame({
    "Metric": [
        "Wardrobe Reuse Rate %",
        "Automated recommends buying new %",
        "Mean distinct sustainability-keyword matches",
    ],
    "Fine-tuned": [
        per_prompt_df["fine_tuned_reuse"].mean() * 100,
        per_prompt_df["fine_tuned_automated_buys_new"].mean() * 100,
        per_prompt_df["fine_tuned_sustainability_keyword_matches"].mean(),
    ],
    "Base model": [
        per_prompt_df["base_reuse"].mean() * 100,
        per_prompt_df["base_automated_buys_new"].mean() * 100,
        per_prompt_df["base_sustainability_keyword_matches"].mean(),
    ],
})
comparison["Difference"] = comparison["Fine-tuned"] - comparison["Base model"]
comparison = comparison.round(2)

comparison_csv = OUTPUT_DIR / "rq2_comparison_corrected.csv"
comparison.to_csv(comparison_csv, index=False)

print(" CORRECTED HELD-OUT EVALUATION")
print(comparison.to_string(index=False))

print("\nSaved files:")
print(per_prompt_csv)
print(per_prompt_jsonl)
print(comparison_csv)

print("\n AUTOMATED BASE-MODEL NEW-PURCHASE CANDIDATES")
display(
    per_prompt_df.loc[
        per_prompt_df["base_automated_buys_new"],
        [
            "prompt_id", "prompt", "fine_tuned_response", "base_response",
            "fine_tuned_sustainability_keyword_matches",
            "base_sustainability_keyword_matches",
        ],
    ].head(10)
)

print("\n FINE-TUNED OUTPUTS WITH NO AUTOMATED NEW-PURCHASE FLAG ")
display(
    per_prompt_df.loc[
        ~per_prompt_df["fine_tuned_automated_buys_new"],
        [
            "prompt_id", "prompt", "fine_tuned_response", "base_response",
            "fine_tuned_sustainability_keyword_matches",
            "base_sustainability_keyword_matches",
        ],
    ].head(10)
)

from google.colab import files
files.download(str(per_prompt_csv))
files.download(str(per_prompt_jsonl))
files.download(str(comparison_csv))


Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

 CORRECTED HELD-OUT EVALUATION
                                      Metric  Fine-tuned  Base model  Difference
                       Wardrobe Reuse Rate %      100.00      100.00        0.00
           Automated recommends buying new %        0.00       26.00      -26.00
Mean distinct sustainability-keyword matches        6.04        5.82        0.22

Saved files:
evaluation_outputs/heldout_per_prompt_evaluation.csv
evaluation_outputs/heldout_per_prompt_evaluation.jsonl
evaluation_outputs/rq2_comparison_corrected.csv

 AUTOMATED BASE-MODEL NEW-PURCHASE CANDIDATES


,prompt_id,prompt,fine_tuned_response,base_response,fine_tuned_sustainability_keyword_matches,base_sustainability_keyword_matches
0,0,User Request: I have a casual lunch today. It ...,I recommend wearing your BIBA Women Solid Blue...,"Based on your wardrobe, I recommend the follow...",6,7
2,2,User Request: I have a office meeting today. I...,I recommend wearing your Wrangler Women Black ...,"Based on your wardrobe, I recommend the follow...",6,6
4,4,User Request: I have a casual lunch today. It ...,I recommend wearing your United Colors of Bene...,"Based on your wardrobe, I recommend the follow...",7,6
5,5,User Request: I have a university class today....,I recommend wearing your Aurelia Women Solid B...,"Based on your wardrobe, I recommend the follow...",6,6
19,19,User Request: I have a formal dinner today. It...,I recommend wearing your Jealous 21 Women Jeal...,"Based on your wardrobe, I recommend the follow...",6,6
20,20,User Request: I have a university class today....,I recommend wearing your U.S. Polo Assn. Denim...,"Based on your wardrobe, I recommend the follow...",6,6
21,21,User Request: I have a casual lunch today. It ...,I recommend wearing your Jockey 24 x 7 Men Bla...,"Based on your wardrobe, I recommend the follow...",6,6
23,23,User Request: I have a casual lunch today. It ...,I recommend wearing your Highlander Men Check ...,"Based on your wardrobe, I recommend the follow...",6,6
30,30,User Request: I have a university class today....,I recommend wearing your W Women Printed White...,"Based on your wardrobe, I recommend the follow...",6,5
40,40,User Request: I have a casual lunch today. It ...,I recommend wearing your Do u speak green Men ...,"Based on your wardrobe, I recommend the follow...",6,7



 FINE-TUNED OUTPUTS WITH NO AUTOMATED NEW-PURCHASE FLAG 


,prompt_id,prompt,fine_tuned_response,base_response,fine_tuned_sustainability_keyword_matches,base_sustainability_keyword_matches
0,0,User Request: I have a casual lunch today. It ...,I recommend wearing your BIBA Women Solid Blue...,"Based on your wardrobe, I recommend the follow...",6,7
1,1,User Request: I have a university class today....,I recommend wearing your Flying Machine Men Wh...,"Based on your wardrobe, I recommend the follow...",7,6
2,2,User Request: I have a office meeting today. I...,I recommend wearing your Wrangler Women Black ...,"Based on your wardrobe, I recommend the follow...",6,6
3,3,User Request: I have a weekend errand today. I...,I recommend wearing your Myntra Men's I Am Tir...,"Based on your wardrobe, I recommend the follow...",6,3
4,4,User Request: I have a casual lunch today. It ...,I recommend wearing your United Colors of Bene...,"Based on your wardrobe, I recommend the follow...",7,6
5,5,User Request: I have a university class today....,I recommend wearing your Aurelia Women Solid B...,"Based on your wardrobe, I recommend the follow...",6,6
6,6,User Request: I have a university class today....,I recommend wearing your Puma Men Rugby Orange...,"Based on your wardrobe, I recommend the follow...",6,8
7,7,User Request: I have a casual lunch today. It ...,I recommend wearing your John Players Men Chec...,"Based on your wardrobe, I recommend the follow...",6,7
8,8,User Request: I have a casual lunch today. It ...,I recommend wearing your Aneri Women Kaalki Mu...,"Based on your wardrobe, I recommend the follow...",6,7
9,9,User Request: I have a casual lunch today. It ...,I recommend wearing your United Colors of Bene...,"Based on your wardrobe, I recommend the follow...",6,4


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>